# RSR Native Adapter v1

目标：把 `rsr_mse` 从 relation-strength Ridge proxy 推进到 `src/model` 的 native temporal relation-strength adapter。Notebook 只保留实验流程、外部参考 URL 和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.graph import RSRGraphRegressor
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'rsr_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'rgcn_mse', 'rsr_mse', 'hypergraph_mse'),
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:8])
estimator, constant_prediction = fit_estimator_for_model('rsr_mse', train=train, feature_columns=feature_columns)
capability = model_capability_table(('rsr_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary()

assert constant_prediction is None
assert isinstance(estimator, RSRGraphRegressor)
assert estimator.feature_count_ == 8
assert estimator.node_count_ == 8
assert estimator.temporal_channel_count_ == 3
assert estimator.relation_strength_.shape == (8,)
assert estimator.relation_gate_.shape == (8,)
assert hasattr(estimator, 'model_')
assert capability['paper_fidelity'] == 'native_graph'
assert capability['is_proxy'] is False
assert model_adapter_name('rsr_mse') == 'rsr_temporal_relation_strength_mse_v1'
assert 'rsr_mse' in fidelity['native_models']
assert 'rsr_mse' not in fidelity['proxy_models']
assert len(fidelity['native_models']) == 24
assert len(fidelity['proxy_models']) == 8

summary = {
    'schema_version': 'research.rsr_native_adapter_v1',
    'status': 'completed',
    'model': 'rsr_mse',
    'adapter': model_adapter_name('rsr_mse'),
    'estimator': estimator.__class__.__name__,
    'node_count': estimator.node_count_,
    'feature_count': estimator.feature_count_,
    'temporal_channel_count': estimator.temporal_channel_count_,
    'hidden_size': estimator.hidden_size,
    'paper_fidelity': capability['paper_fidelity'],
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'paper_reference': {
        'source_id': 'arxiv:1809.09441',
        'url': 'https://arxiv.org/abs/1809.09441',
        'usage': '自有摘要和结构假设；实现只复用 temporal graph convolution 与 relation strength gate 的思想，不复制论文全文或第三方代码。',
    },
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.rsr_native_adapter_v1',
 'status': 'completed',
 'model': 'rsr_mse',
 'adapter': 'rsr_temporal_relation_strength_mse_v1',
 'estimator': 'RSRGraphRegressor',
 'node_count': 8,
 'feature_count': 8,
 'temporal_channel_count': 3,
 'hidden_size': 8,
 'paper_fidelity': 'native_graph',
 'is_proxy': False,
 'full_matrix_native_model_count': 24,
 'full_matrix_proxy_model_count': 8,
 'matrix_card': 'research/rsr_native_adapter_v1/matrix_card',
 'paper_reference': {'source_id': 'arxiv:1809.09441',
  'url': 'https://arxiv.org/abs/1809.09441',
  'usage': '自有摘要和结构假设；实现只复用 temporal graph convolution 与 relation strength gate 的思想，不复制论文全文或第三方代码。'},
 'artifact_links': {'analysis_summary': 'research/rsr_native_adapter_v1/matrix_card/analysis_summary.json',
  'model_comparison': 'research/rsr_native_adapter_v1/matrix_card/model_comparison.csv',
  'model_run_ledger': 'research/rsr_native_adapter_v1/matrix_card/model_run_ledger.csv',
  'manifest': 'research/rsr_native_adapter_v1/

## 外部参考摘要

- Feng et al., `Temporal Relational Ranking for Stock Prediction`：核心方向是把时序表示、股票关系和 ranking signal 结合，用关系强度帮助排序/预测。参考：https://arxiv.org/abs/1809.09441
- 本仓库实现：SOXL 日频监督任务当前没有股票横截面 ranking 标签，本轮先把特征列视为 feature nodes，用 current/lag/delta 三个 temporal channels 表示节点，并用训练窗口目标相关性初始化 relation-strength gate。后续若引入真实股票池和 ranking objective，应替换 label/objective 和 graph provider，而不是把实验参数硬编码进 notebook。

## 中文结论

- 做了什么：在 `src/model.graph` 增加 `RSRGraphRegressor`，把 `rsr_mse` 从 `GraphFeatureRidgeRegressor(graph_kind='rsr')` proxy 升级为 `native_graph`。实现使用 current/lag/delta temporal node channels，并用 relation-strength gate 进行关系传播后回归预测。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`rgcn_mse`、`rsr_mse`、`hypergraph_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- 模型实现状态：全模型注册表当前 native=24、proxy=8；`rsr_mse` 已进入 `native_graph`。
- Regime/区间差异：本轮只证明 RSR native adapter 能进入统一回测和 analysis contract；具体 regime winner/loser 以 `matrix_card/analysis_summary.json` 和 `model_review_briefing.csv` 为准。
- 是否晋升：否。本次是模型实现和链路覆盖升级，不直接晋升生产策略。
- 风险：当前 relation strength 来自单资产监督目标相关性，不是真实股票关系或横截面 ranking；后续需要真实股票池、ranking objective 和更长窗口复核。
- 下一步：刷新全模型 ledger，然后继续替换 `hypergraph_mse`、`thgnn_mse`、`sthan_mse` 等剩余 graph proxy。